<a href="https://colab.research.google.com/github/ThrishaD2/MachineLearning/blob/main/1BM24CS426_Lab_10_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving heart (1).csv to heart (1).csv


In [3]:
# Step 1: Import libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

# Step 2: Load dataset
df = pd.read_csv("heart (1).csv")

# Step 3: Clean column names (IMPORTANT FIX)
df.columns = df.columns.str.strip()

print("Columns in dataset:", df.columns)
print(df.head())

# Step 4: Separate features and target (SAFE METHOD)
X = df.iloc[:, :-1]   # all columns except last
y = df.iloc[:, -1]    # last column as target

# Step 5: Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(exclude=['object']).columns.tolist()

# Step 6: Encoding

# Label Encoding for binary categorical columns
le = LabelEncoder()
for col in categorical_cols:
    if X[col].nunique() == 2:
        X[col] = le.fit_transform(X[col])

# One-Hot Encoding for remaining categorical columns
ct = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

X = ct.fit_transform(X)

# Step 7: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 8: Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -------------------------------
# STEP 9: Train models WITHOUT PCA
# -------------------------------

models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

print("\n--- Accuracy WITHOUT PCA ---")
accuracies_without_pca = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies_without_pca[name] = acc
    print(f"{name}: {acc:.4f}")

# -------------------------------
# STEP 10: Apply PCA
# -------------------------------
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("\nNumber of components after PCA:", pca.n_components_)

# -------------------------------
# STEP 11: Train models WITH PCA
# -------------------------------

print("\n--- Accuracy WITH PCA ---")
accuracies_with_pca = {}

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)
    accuracies_with_pca[name] = acc
    print(f"{name}: {acc:.4f}")

# -------------------------------
# STEP 12: Final Comparison
# -------------------------------

print("\n--- Final Comparison ---")
for model in models.keys():
    print(f"{model}: Without PCA = {accuracies_without_pca[model]:.4f}, With PCA = {accuracies_with_pca[model]:.4f}")

Columns in dataset: Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='object')
   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up        